# 3.1 Scaling laws: what more data buys, and what it asks of the model

At the end of this notebook you can:

- **read a scaling law**: the loss falls as a power of the data, the parameters or the
  compute, a straight line on log-log axes whose slope says what 10× more buys, and only
  inside the range that was measured;
- **say why data and parameters grow together**: for a model that learns from scratch,
  extra parameters buy almost nothing on little data; with the model held fixed, extra data
  buys less and less;
- **measure such a curve yourself** and draw a conclusion from it that goes beyond the top
  score.

**Why this matters.** You have a problem and a few thousand labelled examples. You read a
paper, take its architecture, its size and its training length, and train it on your data.
The paper trained on a thousand times more. The model does not learn, and machine learning
looks hard. The scaling laws say why: what a model of a
given size reaches depends on how much data it gets, and the right size, the right number
of steps and the right amount of data are read off a curve of your own problem. This
notebook measures such curves.

The two slides of lesson 3 on scaling laws showed the ratios of Kaplan et al. (2020) and
Hoffmann et al. (2022). Here you first measure your own curve, then read theirs next to it.

## 3.1.1 The data: Food-101 through a frozen encoder

Food-101 holds photos of 101 dishes, from apple pie to waffles. The cache on the hub holds
one 1024-dim vector per photo from `dinov2-large`, a pretrained image encoder as in
lesson 1, and the label of the dish. The encoder is **frozen**: its weights stay as they
are, and only a head on top of the vectors is trained.

The cache has two splits:

- **train**, 750 photos per class. Food-101's authors did not clean these labels by hand,
  so some photos carry the wrong dish. `food_scaling.py` (next to this notebook) keeps
  50 photos per class of it apart as the **validation** set, and takes the training images
  from the other 700;
- **test**, 250 photos per class, labelled by hand.

As in lesson 2, validation chooses and test judges. Every choice here (the step at which a
run is read) is made on validation; test has the hand-checked labels, so the curves are
read on test.

A training set of 10 images per class takes the first 10 of every class in one fixed order,
so a smaller set is always part of a larger one. The next cell downloads the cache once and
loads it.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import torch
from food_scaling import CLASSES, PER_CLASS, load, train
from notebooktester import param

PLAIN = FuncFormatter(lambda v, _: f"{v:g}")  # 0.6 instead of 6×10⁻¹ on a log axis

food = load()
print(f"train      {tuple(food.X.shape)}   (of which 50 per class are validation)")
print(f"validation {tuple(food.X_valid.shape)}")
print(f"test       {tuple(food.X_held.shape)}")
print(
    f"{CLASSES} classes; test photos per class: {torch.bincount(food.y_held).unique().tolist()}"
)

## 3.1.2 Your own data axis

**Question.** As a head gets more training images, how far does its test loss fall, and
how many optimizer steps does it need to get there?

**Setup.** Lesson 1's linear head, `Linear(1024, 101)`: 1024 · 101 + 101 = 103,525
parameters. It is trained on 10, 30, 70, 200 and 700 images per class: 1,010 to 70,700
images, a factor of 70.

One **optimizer step** is one batch of 128 images: a forward pass, a backward pass and one
update of the weights. Every run trains well past its lowest validation loss; the step of
that lowest point is chosen, and the test loss is read at that step. One seed per size.

In [ ]:
SIZES = param(PER_CLASS, test=PER_CLASS[:3])

runs = {}
for pc in SIZES:
    X, y = food.subset(pc)
    runs[pc] = train(0, X, y, food)  # width 0: the linear head

COLOURS = ["#E69F00", "#56B4E9", "#009E73", "#D55E00", "#0072B2", "#CC79A7"]
plt.figure(figsize=(8, 4.2))
for (pc, r), c in zip(runs.items(), COLOURS):
    curve = r["curve"]
    plt.plot(curve["step"], curve["held"], color=c, label=f"{r['n']:,} images")
    plt.plot(r["best_step"], r["held"], "o", color=c, ms=7)
plt.xscale("log")
plt.yscale("log")
plt.gca().yaxis.set_major_formatter(PLAIN)
plt.gca().yaxis.set_minor_formatter(PLAIN)
plt.ylim(0.18, 1.5)
plt.xlabel("optimizer steps (batches of 128, log)")
plt.ylabel("test loss (cross-entropy, log)")
plt.title("With more images, the chosen step comes later and its test loss lies lower")
plt.legend(frameon=False)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

Every curve starts at its first evaluation, after half a pass over its images, so a larger
set starts further to the right. Both axes are logarithmic: equal distances are equal
factors. The dot is the step chosen on validation; the test loss is read there. The next
cell prints the numbers at the dot, and then what happens when the smallest set is trained
for as many epochs as the largest one needed.

In [ ]:
print(
    f"{'images':>7s} {'step at the dot':>16s} {'epochs':>7s} {'test loss':>10s} {'test accuracy':>14s}"
)
for r in runs.values():
    epochs = r["best_step"] / (r["n"] / 128)
    print(
        f"{r['n']:7,d} {r['best_step']:16d} {epochs:7.1f} {r['held']:10.3f} {r['held_acc']:14.3f}"
    )

# the epochs of the largest run, used on the smallest set
small, large = runs[SIZES[0]], runs[SIZES[-1]]
epochs = large["best_step"] / (large["n"] / 128)
steps = epochs * small["n"] / 128
at = int(np.argmin(np.abs(np.array(small["curve"]["step"]) - steps)))
print(
    f"\n{epochs:.1f} epochs on {small['n']:,} images are {steps:.0f} steps: "
    f"test loss {small['curve']['held'][at]:.3f}, against {small['held']:.3f} at its own dot"
)

**Reading.** From 1,010 to 70,700 images the test loss at the dot falls from 0.482 to
0.225, and the step of the dot moves from 180 to 1,104: more data asks for more steps
before the head has learnt what it can. Counted in passes over the data (epochs) it goes the
other way, from 23 to 2: an epoch over more images holds more steps. Copy the 2 epochs of
the largest run onto the smallest set and the head stops after 16 steps, at a test loss of
1.1 instead of 0.48. A training length taken from someone else's data says little
about yours; your own validation curve says when to stop.

## 3.1.3 A straight line on log-log axes

A **power law** says the loss is a constant times the data to a negative power:
$L(D) = a \cdot D^{-\alpha}$, where $D$ is here the number of training images. Take the log of both sides and it is a straight line,
$\log L = \log a - \alpha \log D$, with slope $-\alpha$. So 10× more data multiplies the
loss by $10^{-\alpha}$: that factor, read off the slope, is what the law tells you.

The next cell draws the test loss at the dot of every run against the number of images,
both on log axes. It fits a straight line through the three smallest sizes only, and then
checks the line on the two larger sizes it has not seen.

The head can only get what the frozen vectors hold. Can the loss keep falling by the same
factor for every 10× more data, and on which side of the line would that put the two
larger sizes?

In [ ]:
D = np.array([r["n"] for r in runs.values()], dtype=float)
L = np.array([r["held"] for r in runs.values()])
slope, intercept = np.polyfit(np.log10(D[:3]), np.log10(L[:3]), 1)
line = lambda d: 10 ** (intercept + slope * np.log10(d))

plt.figure(figsize=(7, 4.2))
d = np.geomspace(D[0], 100_000, 50)
plt.plot(
    d,
    line(d),
    "--",
    color="#D55E00",
    label=f"line through the first three: $L \\propto D^{{{slope:.2f}}}$",
)
plt.plot(D[:3], L[:3], "o", color="k", label="fitted on")
plt.plot(D[3:], L[3:], "o", mfc="white", color="k", label="measured, not fitted on")
plt.xscale("log")
plt.yscale("log")
plt.gca().yaxis.set_major_formatter(PLAIN)
plt.gca().yaxis.set_minor_formatter(PLAIN)
plt.xlabel("training images D (log)")
plt.ylabel("test loss at the dot (log)")
plt.title("The line through the small sizes promises more than the large sizes give")
plt.legend(frameon=False)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(
    f"slope {slope:.3f}: 10x the images multiplies the loss by 10^{slope:.3f} = {10**slope:.2f}"
)
for n, l in zip(D[3:], L[3:]):
    print(f"{int(n):6,d} images: the line says {line(n):.3f}, measured {l:.3f}")

**Reading.** A line through 1,010 to 7,070 images has slope −0.23: 10× the images
multiplies the loss by 0.59. The line says 0.183 at 70,700 images; the run gives 0.225.
Both larger sizes lie above the line: the curve bends away from it. A slope holds inside
the range it was measured on, and a line drawn further is a guess, not a measurement.

## 3.1.4 Kaplan's three laws

Kaplan et al. (2020), *Scaling Laws for Neural Language Models*, measured the same form
over far wider ranges. A **language model** reads a text and predicts the next piece of it,
a **token** (a word or a part of one); the whole model is trained from scratch. Its
score is the **test loss**, the cross-entropy per token on held-out text, in nats (the
natural logarithm, as `nn.CrossEntropyLoss` uses). Three resources:

| law | resource | unit | slope $-\alpha$ | 10× buys | holds when |
|---|---|---|---|---|---|
| $L(N)$ | parameters $N$ | parameters | −0.076 | loss × 0.84 | the data is not the limit |
| $L(D)$ | dataset size $D$ | tokens | −0.095 | loss × 0.80 | the model is large enough, stopped at its best step |
| $L(C)$ | compute $C$ | PF-days | −0.050 | loss × 0.89 | every budget gets the model size that does best with it |

- $N$ leaves out the **embedding table**, the lookup table that turns each token into a
  vector before the model works on it; Kaplan counts only the parameters that compute.
- **Compute** is the arithmetic of training: every multiplication and addition is an
  operation, and training a model with $N$ parameters on $D$ tokens costs about
  $C \approx 6 \cdot N \cdot D$ operations: per parameter and per token about 2 for the
  forward pass and 4 for the backward pass. A **PF-day** is $10^{15}$ operations per second
  for a day, $8.64 \cdot 10^{19}$ operations.

10× the data multiplies the loss of a language model by 0.80, and of your Food-101 head, over
its first three sizes, by 0.59. The same form, another slope: other studies measured it in
translation, image classification and speech recognition (Hestness et al. 2017), and in
image and video generation (Henighan et al. 2020), each with its own slope. A slope belongs
to the problem it was measured on; the slope of your problem is measured on your problem.

## 3.1.5 Why the line bends: data and parameters grow together

In 3.1.3 only the data grew. The model stayed the same: a frozen encoder and a linear head.
Kaplan's equation 1.5 fits the loss of a model with $N$ parameters, trained from scratch on
$D$ tokens, in one formula:

$$L(N, D) = \left[\left(\frac{N_c}{N}\right)^{\alpha_N/\alpha_D} + \frac{D_c}{D}\right]^{\alpha_D}$$

with constants Kaplan fitted (they are in the next cell). Read it in two steps. When $D$ is small, the term $D_c / D$ is
large and the data sets the loss, whatever $N$ is. When $D$ grows without end, that term
goes to zero and what is left is the loss of the model itself: a floor set by $N$.

The next cell draws equation 1.5 against $D$ for three model sizes, 10× apart. On little
data, does the larger model do better? Which line bends first?

In [ ]:
# Kaplan's constants of eq. 1.5; fitted together with N, alpha_D lands at 0.103, near the
# 0.095 of the data law alone in the table above
ND = {"aN": 0.076, "aD": 0.103, "Nc": 6.4e13, "Dc": 1.8e13}


def kaplan_nd(n, d):
    # Kaplan et al. (2020), eq. 1.5: test loss of a model with n parameters on d tokens
    return ((ND["Nc"] / n) ** (ND["aN"] / ND["aD"]) + ND["Dc"] / d) ** ND["aD"]


# Kaplan's runs: WebText2 cut down by up to 1024x (about 2e7 tokens) up to all of it (2.2e10)
MEASURED = (2e7, 2.2e10)
SIZES_N = [1e7, 1e8, 1e9]
tokens = np.unique(np.concatenate([np.geomspace(1e7, 1e12, 100), MEASURED]))
inside = (tokens >= MEASURED[0]) & (tokens <= MEASURED[1])
plt.figure(figsize=(7, 4.2))
plt.axvspan(*MEASURED, color="grey", alpha=0.08, label="where Kaplan's runs lay")
for n, c in zip(SIZES_N, COLOURS[2:]):
    loss = kaplan_nd(n, tokens)
    plt.plot(tokens, loss, color=c, ls="--", lw=1)
    plt.plot(
        tokens[inside], loss[inside], color=c, lw=2, label=f"N = {n:.0e} parameters"
    )
    plt.axhline(kaplan_nd(n, np.inf), color=c, ls=":", lw=1)
plt.plot([], [], "k--", lw=1, label="the formula drawn further")
plt.xscale("log")
plt.yscale("log")
plt.gca().yaxis.set_major_formatter(PLAIN)
plt.gca().yaxis.set_minor_formatter(PLAIN)
plt.xlabel("dataset size D (tokens, log)")
plt.ylabel("test loss, eq. 1.5 (nats per token, log)")
plt.title(
    "On little data the sizes tie; with N fixed, more data bends to a floor (dotted)"
)
plt.legend(frameon=False)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"{'D (tokens)':>11s}" + "".join(f"{f'N = {n:.0e}':>12s}" for n in SIZES_N))
for d in [*MEASURED[:1], 1e9, *MEASURED[1:], np.inf]:
    print(f"{d:11.1e}" + "".join(f"{kaplan_nd(n, d):12.2f}" for n in SIZES_N))

**Reading.** The line is Kaplan's fit; his measured points are the dots of his Figure 9,
and they lay in the grey band, from about $2 \cdot 10^{7}$ tokens (WebText2 cut down
1024×) to all $2.2 \cdot 10^{10}$. There the line is solid; dashed, it is the formula
drawn further.

At the left edge of the band, on $2 \cdot 10^{7}$ tokens, the three models lie on top of
each other: 4.15, 4.11 and 4.11 nats per token for 100× more parameters. On little data, extra
parameters buy almost nothing for a model that learns from scratch; the data is the limit. That is the paper copied onto your
data: the formula puts the floor of the $10^{9}$ model near $10^{11}$ tokens, and on
$2 \cdot 10^{7}$, a five-thousandth of that, it does no better than the small one.

Why it does no worse either: Kaplan regularised every model, with 10% dropout and by
stopping the training once the test loss no longer fell. Without that, the bigger model
would overfit on little data, because it has the room to learn the few training tokens by
heart. Regularised, it ties with the small one: it costs more compute and buys nothing.

To the right the lines part. The smallest model leaves the others between $10^{8}$ and
$10^{9}$ tokens and runs flat into its floor (3.29); the 100× larger model keeps falling to
about $10^{11}$. More data does not get a model below its own floor; only more parameters
move the floor down. So data and parameters grow together.

**How fast the data should grow with the model depends on the question.**

1. *How much data does a bigger model need, so that the data does not hold it back?*
   From equation 1.5: 8× the parameters shrink the model's own term by
   $8^{\alpha_N/\alpha_D} \approx 5$, and the data term keeps pace when $D$ grows by the
   same factor. So 8× the parameters need about 5× the data (the first of the two slides).
2. *With a fixed compute budget $C \approx 6 N D$, how do you split it over model and
   data?* Kaplan's answer: mostly into the model, 8× the parameters with 2.2× the data.
   Under a fixed budget every extra token costs model size, so Kaplan
   found the lowest loss with a large model stopped early, on less data than question 1
   asks. Hoffmann et al. (2022) measured it again and answered: evenly, 8× with 8×. The next cell draws both answers.

In [ ]:
# The split of a compute budget C ≈ 6·N·D along the two studies, each line drawn
# through the model built on it: Kaplan et al. (2020) N ∝ C^0.73, Hoffmann et al. (2022) N ∝ C^0.5.
MODELS = {  # parameters, tokens, label
    "Gopher": (2.8e11, 3.0e11, "Gopher\n280B parameters, 300B tokens"),
    "Chinchilla": (7.0e10, 1.4e12, "Chinchilla\n70B parameters, 1.4T tokens"),
}
budget = np.geomspace(1e22, 1e25, 50)
paths = {
    "Kaplan: mostly into the model": (0.73, "Gopher", "#E69F00"),
    "Hoffmann: model and data evenly": (0.50, "Chinchilla", "#0072B2"),
}

plt.figure(figsize=(7.5, 4.8))
n_axis = np.geomspace(1e8, 1e13, 50)
for cc, text in [(1e21, "C = 1e21"), (5.5e23, "C ≈ 5.5e23"), (1e25, "C = 1e25")]:
    plt.plot(n_axis, cc / (6 * n_axis), ":", color="grey", lw=1)
    d_right = cc / (6 * 1e13)  # where the line leaves the plot on the right
    if d_right > 1e8:
        plt.text(
            9e12, 0.65 * d_right, text, color="grey", fontsize=8, ha="right", va="top"
        )
    else:
        plt.text(1.15 * cc / (6 * 1e8), 1.15e8, text, color="grey", fontsize=8)
for label, (power, model, colour) in paths.items():
    n0, d0, _ = MODELS[model]
    n = n0 * (budget / (6 * n0 * d0)) ** power
    plt.plot(n, budget / (6 * n), color=colour, lw=2, label=label)
for n, d, text in MODELS.values():
    plt.plot(n, d, "o", color="k")
    plt.annotate(text, (n, d), textcoords="offset points", xytext=(8, -14), fontsize=8)
plt.xscale("log")
plt.yscale("log")
plt.xlim(1e8, 1e13)
plt.ylim(1e8, 1e14)
plt.xlabel("parameters N (log)")
plt.ylabel("tokens D (log)")
plt.title("One budget (dotted), two splits")
plt.legend(frameon=False, loc="lower left")
plt.grid(alpha=0.3, which="major")
plt.tight_layout()
plt.show()

for name, (n, d, _) in MODELS.items():
    print(f"{name:>10s}: C = 6·N·D = {6 * n * d:.1e} operations")

**Reading.** A dotted line is one budget: along it, a larger model gets fewer tokens.
Kaplan's split runs almost flat to the right: a bigger budget goes mostly into
parameters, and the tokens grow slowly. Hoffmann's climbs one decade of tokens for every
decade of parameters: model and data grow at the same pace. Gopher was
built along Kaplan's advice: 280 billion parameters on 300 billion tokens. Chinchilla, along
Hoffmann's: 70 billion on 1.4 trillion, on about the same budget. Chinchilla, a quarter of
the size, reached a lower loss and scored better on most of the benchmarks it was tested on. The rule of thumb is to grow the data with the model; how fast is
what a study measured on its problem, not a law of nature.

Your own curve of 3.1.3 held the model fixed too: a frozen encoder with a linear head. It
bends for the reason equation 1.5 gives: with the model fixed, more data runs into the
model's floor. Here that floor is what the 1024 numbers of the frozen encoder hold about a
dish. What they do not tell apart, no amount of data teaches the head; some wrong labels
among the training photos lift the floor a little more. Only a model that can take in more
moves it, and here that would mean training the encoder itself. The next section turns it
around: it makes the head the limit, so you see the parameter side.

## 3.1.6 A bottleneck whose width you choose

To see the parameter side, you need a model whose size limits it. The frozen encoder's
vectors already separate the 101 dishes almost with a straight line, so a bigger head has
little left to learn; training the encoder itself (300 million parameters, 142 million
images) is beyond this lesson.

So this section builds the limit in: a **bottleneck**, a hidden layer of $w$ units between
the vector and the 101 dishes (`Linear(1024, w)`, GELU, `Linear(w, 101)`). Everything the
head knows about a photo passes through those $w$ numbers, so $w$ sets how much the model
can use: with 16 units, 101 dishes have to be told apart by 16 numbers.

**Question.** What does each doubling of the bottleneck buy, and where does it stop?

**Setup.** Widths 16, 32, 64, 128 and 256, on the same five sizes, with the same rule
as 3.1.2: one learning rate for all, no regularisation besides the choice of the step.
Narrower heads of 4 or 8 units are still learning when the step budget runs out at this
learning rate, so their number would measure the budget, not the width. The
linear head of 3.1.2 has no bottleneck: it passes all 1024 numbers on, so it is drawn as the
reference the widths approach. The 30 runs take about a minute, so they run in
`food_scaling.py` and store `results/food_scaling.json`; the next cell loads it.

In [ ]:
# True: run the sweep of food_scaling.py again (about a minute) before loading it.
# Under notebooktester it runs in a test mode of seconds and stores its result in results/test.
RERUN = param(False, test=True)
RESULTS = Path("results", param("", test="test"))
if RERUN:
    subprocess.run(
        [sys.executable, "food_scaling.py", *param([], test=["--test"])], check=True
    )
sweep = json.loads((RESULTS / "food_scaling.json").read_text(encoding="utf-8"))

widths = sorted({r["width"] for r in sweep})
sizes = sorted({r["n"] for r in sweep})
fig, (ax, steps_ax) = plt.subplots(1, 2, figsize=(12, 4.4))
plt.sca(ax)
for w, c in zip(widths, ["k", *COLOURS]):
    rs = sorted([r for r in sweep if r["width"] == w], key=lambda r: r["n"])
    label = "linear, no bottleneck" if w == 0 else f"{w} units"
    style = dict(ls="--", lw=1.5) if w == 0 else dict(ls="-", lw=1.5)
    plt.plot(
        [r["n"] for r in rs],
        [r["held"] for r in rs],
        marker="o",
        color=c,
        label=label,
        **style,
    )
plt.xscale("log")
plt.yscale("log")
plt.gca().yaxis.set_major_formatter(PLAIN)
plt.gca().yaxis.set_minor_formatter(PLAIN)
plt.xlabel("training images (log)")
plt.ylabel("test loss at the dot (log)")
plt.title("Each doubling of the bottleneck buys less than the one before")
plt.legend(frameon=False, fontsize=8, ncol=2)
plt.grid(alpha=0.3, which="both")

# right: the validation loss of every bottleneck on the smallest size, the dot at the chosen step
plt.sca(steps_ax)
for w, c in zip(widths[1:], COLOURS):
    r = next(r for r in sweep if r["width"] == w and r["n"] == sizes[0])
    i = r["curve"]["step"].index(r["best_step"])
    plt.plot(r["curve"]["step"], r["curve"]["valid"], color=c, lw=1.5)
    plt.plot(r["best_step"], r["curve"]["valid"][i], "o", color=c, ms=7)
plt.xscale("log")
plt.yscale("log")
plt.gca().yaxis.set_major_formatter(PLAIN)
plt.gca().yaxis.set_minor_formatter(PLAIN)
plt.ylim(0.6, 2)
plt.xlabel("optimizer steps (log)")
plt.ylabel("validation loss (log)")
plt.title(f"On {sizes[0]:,} images, a wider bottleneck reaches its best step sooner")
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"{'head':>10s} {'parameters':>11s}" + "".join(f"{n:>9,d}" for n in sizes))
for w in widths:
    rs = sorted([r for r in sweep if r["width"] == w], key=lambda r: r["n"])
    name = "linear" if w == 0 else f"{w} units"
    print(
        f"{name:>10s} {rs[0]['params']:>11,d}"
        + "".join(f"{r['held']:9.3f}" for r in rs)
    )

print("\nchosen step (lowest validation loss)")
for w in widths:
    rs = sorted([r for r in sweep if r["width"] == w], key=lambda r: r["n"])
    name = "linear" if w == 0 else f"{w} units"
    print(f"{name:>10s} {'':>11s}" + "".join(f"{r['best_step']:9,d}" for r in rs))

**Reading.** Read down a column of the table. At 20,200 images, 16 → 32 units takes 0.103 off
the loss, 32 → 64 takes 0.049, 64 → 128 takes 0.014 and 128 → 256 only 0.004: with the data
fixed, each doubling takes less off than the one before, and the widths close in on the
linear head without passing it. Every column shows the same: the bend of 3.1.3 mirrored,
with the data fixed instead of the model. Past 128 units the head is no longer what limits
the loss; the encoder's vectors are. At 70,700 images the 256-unit head even lands a little
above 128 units (0.251 against 0.238). That is one run each, and a difference that small is
as likely the noise of a run as a property of the width.

Now read the chosen steps, and the right panel. On 1,010 images the 16-unit head reaches its
lowest validation loss after 327 steps, the 256-unit head after 105. The wider head fits
the few training photos faster, and after its dot its validation loss rises: it overfits
sooner. (Validation lies above test throughout: its labels were not checked by hand.) Stopping at the dot is the regularisation that holds it there. With it, 256 units
do barely better than 128 at twice the parameters: past 128, extra width is mostly compute
spent.

Now compare the columns. At 1,010 images, 16 and 64 units lie 0.41 apart (0.987 against
0.580); at 70,700 only 0.10 (0.350 against 0.253). The widths lie furthest apart on little
data and close in as the data grows: the opposite of equation 1.5. Kaplan's models learn from scratch, so on little data
none of them has learnt much, and they tie. These heads sit on a pretrained encoder whose
vectors already separate the dishes. A wide bottleneck passes most of that on from the first
thousand photos; a narrow one has to find which few of the 1024 directions to keep, and more
photos help it find them. So "on little data a bigger model buys nothing" holds for a model
that learns from scratch. On top of a pretrained encoder, the small head is the one that
needs the data.

## 3.1.7 Beyond the top score

**Question.** The top score is the linear head on all 70,700 images: 0.935 test accuracy.
What does the curve say beyond that number?

**Reading, from the table of 3.1.2.** 20,200 images give 0.923: 3.5× fewer labelled photos
for 1.2 points of accuracy, and a best step of 468 instead of 1,104. Labelling is often the
most expensive part of a problem. Measured on parts of the labels you have, the curve shows
what more labels bought so far. Whether 1.2 points are
worth 50,500 more labels is a choice for whoever pays for the labels; the curve gives the
price.

A smaller head would not make an answer cheaper: the encoder runs its 300 million
parameters for every photo, and every head here is small next to it. On this problem the
saving lies in labels and steps.

## What you now know

- A **scaling law** says the loss falls as a power of the data, the parameters or the
  compute: a straight line on log-log axes, whose slope says what 10× more buys. Every problem
  has its own slope: measure yours.
- A line holds inside the range it was measured on; drawn further, it promised more than
  the runs gave.
- **Data and parameters grow together.** For a model that learns from scratch, a bigger
  model buys nothing on little data; with the model fixed, more data runs into the model's floor, and only a bigger model moves it.
  How fast the data should grow is a finding of a study and its question: Kaplan put most
  of a budget into the model, Hoffmann split it evenly, and Hoffmann's smaller Chinchilla
  did better.
- **A model too big for its data** risks overfitting; one too small cannot overfit.
  Regularised (Kaplan: dropout and early stopping; here: stopping at the best step), the
  big one does not overfit, and its extra parameters are mostly compute spent.
- More data asks for more optimizer steps before the best test loss; a number of epochs
  from another dataset says little about yours.
- **A bottleneck** on a frozen encoder: with the data fixed, each doubling of its width
  takes less off the loss than the one before, until the encoder, not the head, is the
  limit. On little data the narrow head lags most,
  because the pretrained vectors already did most of the learning.
- **Beyond the top score:** the curve gives the price of more labels, and whoever pays for
  them decides.

**When you do this for real.** A paper's model, size and training length come with the
paper's data. With a hundred or a thousand times less, measure your own curve: a few sizes
of the data, a few sizes of the model, each trained past its best step. Then choose the
model, the steps and the data from that curve, and dare to start small.

## Exercises

**E1: a floor in the fit.** Fit $L(D) = L_\infty + a \cdot D^{-\alpha}$ to the five points of
3.1.3 instead of a straight line (`scipy.optimize.curve_fit` fits a formula with free
constants to points). How large is the floor $L_\infty$? Extrapolate both fits to 700,000
images: which would you believe, and why can neither be checked on this data?

**E2: your own slope on another dataset.** Repeat 3.1.2 and 3.1.3 on DTD
(`VectorCache.from_hub("pttrn-io/dtd-dinov2-large")`: photos of 47 textures, 80 per class
in train and 40 in test), through the same frozen `dinov2-large`. Take, say, 10 per class
apart for validation and train on 5 to 70 per class. Is its slope steeper or flatter than
Food-101's, and what about textures and dishes could make the difference?

**E3: a curve of your own choice.** Measure a scaling curve along an axis you pick: the
width of the bottleneck on DTD, the size of the encoder (`pttrn-io/dtd-dinov2-small`,
`-base` and `-large` hold the same photos through three sizes of `dinov2`), or a dataset of
your own. What does your curve say beyond the top score?